In [1]:
# Import libraries and configure directories:
import pandas as pd
import shutil
from pathlib import Path

# Kaggle directories
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

# Required Spotify datasets
SOURCE_FILES = [
    "charts_songs_daily.csv",
    "charts_artists_daily.csv",
    "charts_albums_weekly.csv",
    "artists.csv"]

# Automatically locate each dataset
sources = {}

for filename in SOURCE_FILES:

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Dataset not found: {filename}")

    sources[filename] = matches[0]

    print(f"Found: {sources[filename]}")

print("\nAll source datasets verified.")

Found: /kaggle/input/datasets/gonzalopezgil/spotify-charts-daily-updated/charts_songs_daily.csv
Found: /kaggle/input/datasets/gonzalopezgil/spotify-charts-daily-updated/charts_artists_daily.csv
Found: /kaggle/input/datasets/gonzalopezgil/spotify-charts-daily-updated/charts_albums_weekly.csv
Found: /kaggle/input/datasets/gonzalopezgil/spotify-charts-daily-updated/artists.csv

All source datasets verified.


In [2]:
# Cereate the daily dataset extraction function:
def extract_daily_data(source_name, output_name):

    # Retrieve the source path from Cell 1:
    source_path = sources[source_name]
    output_path = OUTPUT_DIR / output_name

    latest_date = None

    # First pass: identify the latest available date:
    for chunk in pd.read_csv(
        source_path,
        usecols=["date"],
        chunksize=100_000):
        dates = pd.to_datetime(
            chunk["date"],
            errors="coerce")

        chunk_max = dates.max()

        if pd.notna(chunk_max):
            if latest_date is None or chunk_max > latest_date:
                latest_date = chunk_max

    if latest_date is None:
        raise ValueError(f"No valid dates in {source_name}")

    # Calculate the latest 365 calendar days:
    start_date = latest_date - pd.Timedelta(days=364)

    total_rows = 0
    first_chunk = True

    # Second pass: filter and export matching records:
    with output_path.open(
        "w",
        encoding="utf-8",
        newline="") as output:

        for chunk in pd.read_csv(
            source_path,
            chunksize=100_000):
            dates = pd.to_datetime(
                chunk["date"],
                errors="coerce")

            filtered = chunk.loc[
                dates.between(start_date, latest_date)]

            filtered.to_csv(
                output,
                index=False,
                header=first_chunk)

            first_chunk = False
            total_rows += len(filtered)

    print(f"Created: {output_name}")
    print(f"Start date: {start_date.date()}")
    print(f"End date: {latest_date.date()}")
    print(f"Total records: {total_rows:,}")
    print()

In [3]:
# Extract the latest 365 days of song charts:
extract_daily_data(
    source_name="charts_songs_daily.csv",
    output_name="charts_songs_daily_365D.csv")

Created: charts_songs_daily_365D.csv
Start date: 2025-11-13
End date: 2026-11-12
Total records: 5,124,513



In [4]:
# Extract the latest 365 days of artist charts:
extract_daily_data(
    source_name="charts_artists_daily.csv",
    output_name="charts_artists_daily_365D.csv")

Created: charts_artists_daily_365D.csv
Start date: 2025-11-13
End date: 2026-11-12
Total records: 4,923,366



In [5]:

# Retrieve the weekly album source file:
source_path = sources["charts_albums_weekly.csv"]

# Define the output file:
output_path = OUTPUT_DIR / "charts_albums_weekly_52W.csv"

# Store available weekly chart dates:
all_dates = set()

# First pass: collect distinct chart dates:
for chunk in pd.read_csv(
    source_path,
    usecols=["date"],
    chunksize=100_000
):
    dates = pd.to_datetime(
        chunk["date"],
        errors="coerce"
    ).dropna()

    all_dates.update(dates.unique())

# Select the latest 52 distinct chart dates:
latest_weeks = set(
    sorted(all_dates, reverse=True)[:52])

if not latest_weeks:
    raise ValueError("No valid album chart dates found.")

total_rows = 0
first_chunk = True

# Second pass: extract selected chart weeks
with output_path.open(
    "w",
    encoding="utf-8",
    newline="") as output:

    for chunk in pd.read_csv(
        source_path,
        chunksize=100_000):
        dates = pd.to_datetime(
            chunk["date"],
            errors="coerce")

        filtered = chunk.loc[
            dates.isin(latest_weeks)]

        filtered.to_csv(
            output,
            index=False,
            header=first_chunk)

        first_chunk = False
        total_rows += len(filtered)

print("Created: charts_albums_weekly_52W.csv")
print(f"Chart weeks: {len(latest_weeks)}")
print(f"Total records: {total_rows:,}")

Created: charts_albums_weekly_52W.csv
Chart weeks: 52
Total records: 526,800


In [6]:
# Retrieve the original artist master dataset:
source_path = sources["artists.csv"]

# Define the output file:
output_path = OUTPUT_DIR / "artists.csv"

# Copy the entire dataset:
shutil.copy2(
    source_path,
    output_path)

print("Created: artists.csv")
print(f"File size: {output_path.stat().st_size / (1024 ** 2):.2f} MB")

Created: artists.csv
File size: 4.22 MB


In [7]:
# Define the expected output files:
OUTPUT_FILES = [
    "charts_songs_daily_365D.csv",
    "charts_artists_daily_365D.csv",
    "charts_albums_weekly_52W.csv",
    "artists.csv"]

# Verify each extracted dataset:
for filename in OUTPUT_FILES:

    file_path = OUTPUT_DIR / filename

    # Check whether the file exists:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Missing output: {filename}")

    # Check file size:
    size_mb = file_path.stat().st_size / (1024 ** 2)

    # Check whether the CSV contains a header and records:
    with file_path.open("r", encoding="utf-8") as file:
        header = file.readline()
        first_record = file.readline()

    if not header or not first_record:
        raise ValueError(
            f"Empty dataset: {filename}")

    print(f"Validated: {filename} ({size_mb:.2f} MB)")

print("\nAll four datasets validated successfully.")

Validated: charts_songs_daily_365D.csv (1122.73 MB)
Validated: charts_artists_daily_365D.csv (547.52 MB)
Validated: charts_albums_weekly_52W.csv (102.15 MB)
Validated: artists.csv (4.22 MB)

All four datasets validated successfully.


In [8]:
# Create a ZIP archive for your project:
from zipfile import ZipFile, ZIP_DEFLATED

# Define the ZIP output path:
zip_path = OUTPUT_DIR / "spotify_raw_data.zip"

# Create the ZIP archive:
with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:

    for filename in OUTPUT_FILES:

        file_path = OUTPUT_DIR / filename

        # Preserve the existing project folder structure:
        archive.write(
            file_path,
            arcname=f"data/raw/{filename}")

print(f"ZIP created: {zip_path.name}")
print("All four CSV files included.")

ZIP created: spotify_raw_data.zip
All four CSV files included.
